[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/convection/convection.ipynb)

# Rayleigh–Bénard Convection

This example computes Rayleigh–Bénard convection: a layer of liquid between a warm floor and a cool ceiling, which conducts heat up while it stays still and turns over in rolls once the warm liquid's buoyancy beats its viscosity and its conduction of heat. Warm liquid is lighter, so the still layer is top-heavy; a parcel that rises keeps its buoyancy only if it rises before its warmth has conducted away, and the Rayleigh number compares the two times. Here the layer turns over from a slightly disturbed, conducting start, as [Bergé and Dubois (1984)](https://doi.org/10.1080/00107518408210730) set out. As it does, it trades the potential energy of its top-heavy liquid for the kinetic energy of its rolls, which viscosity turns into heat, and the example keeps that budget exactly.

The algebra is the plane joined with three degenerate directions, for volume, mass and entropy, `Algebra("x+y+") * Algebra("v0m0s0")`. A parcel of the liquid is a point weighed by its volume, offset along the mass's direction by its mass and along the entropy's by its entropy, `volume + (density - expansion * warmth) * mass + warmth * entropy`: a unit volume holds a little less mass the warmer it is. A drive is minus the gradient of a potential wedged with what it acts on, and does its work on the flux of that quantity. Gravity's drive, the geopotential's gradient wedged with mass, pulls on the mass a flow carries, and since the still layer's weight is held by the pressure, only a warm parcel's missing mass is left to pull on. The temperature's drive, its gradient wedged with entropy, does no work on the flow: a conductivity maps it to the entropy's flux, which leaves each cell as the entropy the flow carries does. The flow is a stream function on the mesh's vertices; its mass resists its change and its viscosity its turning, and the pressure, which only keeps the flow from filling or emptying any cell, does no work on it and drops out. Read on a flux of parcels, `flows & parcels`, gravity's drive gives the work it does on the flow, and read on the parcels' places joined with them, `places & parcels`, minus their potential energy: one pairing, so that what the flow gains the parcels' potential energy loses. The momentum's own advection is left out, as the large Prandtl number of water and oils makes it small.

In vector-calculus notation, the Oberbeck–Boussinesq equations read $\partial_t\mathbf v = -\nabla p/\rho_0 + \nu\nabla^2\mathbf v + g\alpha T\,\hat{\mathbf z}$, $\nabla\cdot\mathbf v = 0$ and $\partial_t T + \mathbf v\cdot\nabla T = \kappa\nabla^2 T$, here without the term $(\mathbf v\cdot\nabla)\mathbf v$; the Rayleigh number is $\mathrm{Ra} = g\alpha\,\Delta T\,d^3/(\nu\kappa)$, and between rigid plates the still layer gives way at $\mathrm{Ra}_c \approx 1707$, to rolls of wavenumber $a_c \approx 3.117$ times the inverse depth.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Callable, Iterator
from itertools import islice

import numpy as np

from numga import Algebra, NumpyContext, stack
from numga.sparse import SparseExtensor, spdiag
from examples.util.animation import show_states
from examples.fluids.convection import render, scenarios
from examples.fluids.convection.core import layer
from examples.util.topology import Topology

# The plane, and the volume, the mass and the entropy a parcel holds: each a direction of its own, degenerate, since
# nothing measures a volume, a mass or an entropy against a length.
ga = Algebra("x+y+") * Algebra("v0m0s0")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Point = ga.gatype.antivector()                                                # Antivector-
Plane = ga.gatype.vector()                                                    # Vector+
# The planes reading a place's run along the layer and up it.
Upright = ga.gatype.from_blades("x y")                                        # Vector+
Drive = ga.gatype.from_blades("xv yv xm ym xs ys")                            # Bivector+
Flux = ga.gatype.antibivector()                                               # Antibivector-
# The layer's two directions, along its length and up its depth, and the planes reading a direction's run along each.
length = mv.x.dual()                                                          # [] Point-
height = mv.y.dual()                                                          # [] Point-
directions = stack((length, height))                                          # [2] Point-
readings = stack((mv.x, mv.y))                                                # [2] Plane+
# A unit of volume, the weight of a place, a unit of mass and a unit of entropy.
volume = mv.v.dual()                                                          # [] Point-
mass = mv.m.dual()                                                            # [] Point-
entropy = mv.s.dual()                                                         # [] Point-

## The layer

The layer is a strip of triangles one depth high, closed on itself along its length so that it has a floor and a ceiling but no ends. Its geometry is read on the same strip opened at its seam, its last column the first again a period further along, so that no edge needs telling that it crosses the seam: the two share their triangles, in the same order, and a field on the closed layer reads on the open one. Lengths are in depths, times in the time heat takes to conduct across the layer, and temperatures in the difference between floor and ceiling, a degree.

In [ ]:
# The layer, four depths long, closed and opened at its seam, the maps reading a field on each at the other's vertices,
# and the open layer's vertices, places of unit volume.
closed, opened, closing, opening, places = layer(scenarios.n_along, scenarios.n_across, scenarios.period)
vertices = (volume + mv(Upright, places).dual()).field()                      # Point-[opened vertices]
render.draw_layer(opened, vertices, scenarios.period);

The layer's triangles lie between its warm floor, in red, and its cool ceiling, in blue.

## Turning over

The liquid's flow is a stream function, whose rise at a corner carries the liquid along the facing side. Its kinetic energy is the momentum paired with the velocity, and its conduction of heat is the outflow of the entropy flux the temperature's drive conducts, `mv.s & (gradients ^ conductivity(drives))`; over a triangle the two give the same numbers, the one as a flow's energy, the other as a flux. Gravity's drive does work on the flows the stream function can make by pulling on the mass they carry. The liquid has water's Prandtl number, its viscosity seven times its conduction of heat, and the layer starts still and conducting, with a small disturbance. The cells below take a few seconds, most of it stepping the liquid and drawing the animation.

In [ ]:
# The liquid: its density, its viscosity, the Prandtl number in these units, the mass a unit volume loses per degree,
# and gravity, which with them makes the Rayleigh number.
density, viscosity, expansion = scenarios.density, scenarios.prandtl, scenarios.expansion
gravity = scenarios.rayleigh * viscosity / expansion
# The layer's absolute temperature, water's near 300 K, in degrees of the layer's one-kelvin difference, and the heat a
# unit volume of water holds per degree in the units of the flow's energy, for a layer a centimetre deep.
reference_temperature, heat_capacity = scenarios.reference_temperature, scenarios.heat_capacity
# The step, the steps run, the steps between frames, and the steps before the rolls show, run but not drawn.
dt, n_steps, n_stride, n_unshown = scenarios.dt, scenarios.n_steps, scenarios.n_stride, scenarios.n_unshown

In [ ]:
# The layer's geometry: its triangles read on the open layer and gathered onto the closed one's vertices.
sides = opened.boundary[2, 1] * (opened.d[0] * vertices)                      # [corners] Point-[faces]
# Twice a triangle's area: two of its sides joined, at unit volume, mass and entropy.
double_areas = (((sides[0] & sides[1]) & volume) & mass) & entropy            # Scalar-[faces]
areas = double_areas.norm() / 2                                               # Scalar+[faces]
# A corner's hat gradient: the plane through the facing side, along volume, mass and entropy, over twice the
# area, so that it reads one from the side to the corner.
gradients = (volume & mass & entropy & sides) / double_areas                  # [corners] Plane+[faces]
corners = closed.incidence[2, 0]                                              # [corners] Scalar+[faces <- vertices]
# Each vertex's share of the area around it, and each triangle's mean over its corners.
cells = (~corners * (areas / 3)).sum(axis=0)                                  # Scalar+[vertices]
corner_mean = corners.mean(axis=0)                                            # Scalar+[faces <- vertices]
# A stream function rising by one at a corner carries the liquid along the facing side, half the side's worth
# over the triangle: each corner's flow, and its velocity.
flows = sides / 2                                                             # [corners] Point-[faces]
velocities = flows / areas                                                    # [corners] Point-[faces]
# A stream function's velocity on each triangle, its corners' velocities weighed by its values there, and what
# a flux carries out of each corner's cell: its hat gradient wedged with the flux, gathered onto the vertex.
stream_velocity = (spdiag(velocities * Scalar) * corners).sum(axis=0)         # (Point- <- Scalar+)[faces <- vertices]
cell_outflow = (~corners * spdiag(areas * (gradients ^ Flux))).sum(axis=0)    # (Point- <- Flux-)[vertices <- faces]

In [ ]:
# The liquid's physics on the layer's triangles: its forms, and gravity's drive.
# The kinetic form: the liquid's inertia reads a velocity's run along and up into its momentum, paired with the
# velocity of each other corner over the triangle, and gathered onto the vertices.
inertia = density * (readings * (readings & Point)).sum(axis=0)               # [] Plane+ <- Point-
local = areas * (inertia(velocities[:, None]) & velocities[None, :])          # [corners, corners] Scalar+[faces]
kinetic = (~corners[:, None] * spdiag(local) * corners[None, :]).sum(axis=0).sum(axis=0)   # Scalar+[vertices <- vertices]
# The vorticity at each vertex, the kinetic form's amount over its share of the area and the density, and the
# viscous form, its square over the layer.
vorticity = spdiag(1 / (density * cells)) * kinetic                           # Scalar+[vertices <- vertices]
viscous = viscosity * ~vorticity * spdiag(cells) * vorticity                  # Scalar+[vertices <- vertices]
# The temperature's drive at each corner's hat, minus its gradient wedged with the entropy it pushes, and the
# conductivity, the drive read across each of the layer's directions driving the entropy along it.
drives = -(gradients ^ mv.s)                                                  # [corners] Drive+[faces]
conductivity = ((directions & entropy) * ((directions & entropy) & Drive)).sum(axis=0)   # [] Flux- <- Drive+
# What a flux carries out of each corner's cell: its hat gradient wedged with the flux, read for its entropy,
# gathered onto the vertices.
local = areas * (mv.s & (gradients[:, None] ^ conductivity(drives[None, :]))) # [corners, corners] Scalar+[faces]
conduction = (~corners[:, None] * spdiag(local) * corners[None, :]).sum(axis=0).sum(axis=0)   # Scalar+[vertices <- vertices]
# Gravity's drive, minus the gradient of its potential wedged with mass, and its reading of a degree of warmth at each
# vertex: the entropy the warmth adds and the mass it takes away, joined with the vertex's place.
pull = -(gravity * mv.ym)                                                     # [] Drive+
places = closing * vertices                                                   # Point-[vertices]
lift = pull & (places & (entropy - expansion * mass))                         # Scalar+[vertices]

In [ ]:
def holding(form: SparseExtensor, interior: SparseExtensor) -> Callable[[Scalar, Scalar], Scalar]:
    """The step of a map on a mesh's vertices solved with its walls held: given the right-hand side and the walls'
    values `[...] Scalar+[vertices]`, the values the map takes to it at the free vertices, read by `interior`, and the
    walls' values elsewhere. The map on the free vertices is built once, so that every step only substitutes."""
    free = (interior * form * ~interior) * Scalar                             # (Scalar+ <- Scalar+)[free <- free]
    return lambda rhs, walls: walls + ~interior * free.solve(interior * (rhs - form * walls))


# The free vertices, a field's values at them, nothing at the walls, and each half step's implicit map, solved with
# the walls held: the flow's inertia with its viscosity, and the cells' heat with its conduction.
interior = closed.reading(closed.interior[0])                                 # Scalar+[free <- vertices]
held = ~interior * interior                                                   # Scalar+[vertices <- vertices]
flow_step = holding(kinetic / (dt / 2) + viscous, interior)
heat_step = holding(spdiag(cells / (dt / 2)) + conduction, interior)
# The still layer's temperature, disturbed, and the temperature the walls hold.
start = scenarios.conducting(closing * (mv.y & vertices), scenarios.disturbance)   # Scalar+[vertices]
wall_temperature = start - held * start                                       # Scalar+[vertices]

In [ ]:
def convecting(temperature: Scalar) -> Iterator[tuple[Scalar, Scalar, tuple[Scalar, Scalar, Scalar]]]:
    """The layer's stream function and temperature `Scalar+[vertices]` every `n_stride` steps, from rest and this
    temperature, with the energy the steps so far dissipated, the potential energy their heat moved besides the flow's
    carrying, and the entropy they produced, `Scalar+` each: each step gravity's drive does work on the flows the
    stream function can make, the liquid's mass and viscosity resist them, and the flow carries the entropy along
    while the temperature's drive conducts it, all at the step's midpoint."""
    # The liquid at rest; the midpoint temperature of the step before the first, the start's; and nothing yet
    # dissipated, moved or produced.
    stream = 0 * temperature                                                  # Scalar+[vertices]
    middle_temperature = temperature                                          # Scalar+[vertices]
    dissipated = conducted = produced = 0 * temperature.sites.sum()           # [] Scalar+
    for step in range(n_steps + 1):
        if step % n_stride == 0:
            yield stream, temperature, (dissipated, conducted, produced)
        # The step's midpoint temperature, foreseen as far past this state as the last step's midpoint lay before it.
        middle_temperature = 2 * temperature - middle_temperature
        # Each triangle's parcels per volume at the midpoint: unit volume, holding less mass the warmer they are, and
        # their entropy.
        warmth = corner_mean * middle_temperature                             # Scalar+[faces]
        parcels = volume + (density - expansion * warmth) * mass + warmth * entropy   # Point-[faces]
        # Gravity's drive does work on each corner's flow by pulling on the mass it carries, and the flow's inertia and
        # viscosity take it to the step's midpoint.
        load = (~corners * (pull & (flows & parcels[..., None]))).sum(axis=-1)   # Scalar+[vertices]
        middle_stream = flow_step(kinetic / (dt / 2) * stream + load, 0 * stream)
        # The midpoint's flow carries the same parcels' entropy along; what its flux carries out of each corner's cell
        # leaves it, and the temperature's drive conducts the rest.
        velocity = stream_velocity(middle_stream)                             # Point-[faces]
        flux = velocity & (warmth * entropy)                                  # Flux-[faces]
        carried = mv.s & cell_outflow(flux)                                   # Scalar+[vertices]
        middle_temperature = heat_step(cells / (dt / 2) * temperature - carried, wall_temperature)
        # The step's budget at its midpoint: the energy viscosity dissipates; the potential energy the heat moves
        # besides the flow's carrying, read off each cell's outflow of entropy, nothing at the walls, which hold their
        # temperature, less what the flow carries out; and the entropy viscosity and conduction produce at the layer's
        # absolute temperature, conduction carrying heat by the liquid's capacity for it.
        dissipation = dt * (middle_stream * (viscous * middle_stream)).sites.sum()   # [] Scalar+
        exchange = held * (conduction * middle_temperature + carried) - carried   # Scalar+[vertices]
        dissipated = dissipated + dissipation
        conducted = conducted + dt * (lift * exchange).sites.sum()
        produced = produced + dissipation / reference_temperature + dt * heat_capacity * (
            middle_temperature * (conduction * middle_temperature)).sites.sum() / reference_temperature ** 2
        # The step ends as far past the midpoint as it began before it.
        stream, temperature = 2 * middle_stream - stream, 2 * middle_temperature - temperature

In [ ]:
# Drawn as each frame comes, once the rolls begin to show; the states drawn come back.
states = show_states(islice(convecting(start), n_unshown // n_stride, None), render.animate, opened, vertices, opening,
                     scenarios.period, name="convection", duration_ms=scenarios.frame_duration_ms,
                     n_frames=(n_steps - n_unshown) // n_stride + 1)

The layer is coloured by its temperature, red above the middle's and blue below, with the streamlines in black at a fixed spacing of the stream function, so that the closer they lie the faster the liquid flows. The animation starts as the disturbance's rolls begin to show in the streamlines, while the layer still conducts. The rolls grow, warm liquid rising and cool liquid sinking between them, and overshoot: they turn the warm liquid right over, mixing the layer's middle until the top-heaviness that drives them is spent, and nearly stall. Conduction builds the warm layer at the floor and the cool one at the ceiling up again, the rolls pick up, and after a few swings they carry the heat up steadily; the temperature then changes only in thin layers at the floor and the ceiling, where conduction hands the heat to the flow and takes it back.

## The flow's growth

Read off the states the animation drew, the flow's squared size over the layer rises straight on a logarithmic scale while the rolls grow at a steady rate, overshoots as they turn the layer over, and swings about its settled size as they settle.

In [ ]:
# The flow's squared size over the layer, at each state drawn, and the times of the states.
sizes = stack(tuple((cells * stream.squared()).sites.sum() for stream, _, _ in states))   # [frames] Scalar+
times = (n_unshown + n_stride * np.arange(len(states))) * dt
render.draw_turnover(times, sizes);

## The energy budget

As the layer turns over, the potential energy of its top-heavy liquid becomes the kinetic energy of the rolls, which viscosity dissipates, while the heat moves potential energy of its own, conducted within the layer and taken up by the walls, which hold their temperature. The work gravity's drive does on the flow and the potential energy the flow's carrying releases cancel whenever the force and the carrying read the same flow and the same temperature, whatever they are. Each step reads both with the flow and the temperature of its midpoint, and the flow, stepped as far past its midpoint as it began before it, changes its kinetic energy by just the work done there, so the budget closes to round-off at every step. The viscous form and the conduction are pulled back from positive materials, so the energy dissipated and the entropy produced, at the layer's absolute temperature, never fall.

In vector-calculus notation, the exchange is the identity $\int g\alpha T\,v_z\,dA = -\int g\alpha z\,\mathbf v\cdot\nabla T\,dA$ for a flow with $\nabla\cdot\mathbf v = 0$ running along the walls, and the entropy is produced at the rate $\int \big(\rho_0\nu|\boldsymbol\omega|^2/T_0 + \rho_0 c_p\kappa|\nabla T|^2/T_0^2\big)\,dA$.

In [ ]:
# The states drawn, and what the steps to each dissipated, moved with the heat and produced as entropy.
drawn_streams = stack(tuple(stream for stream, _, _ in states))               # [frames] Scalar+[vertices]
drawn_temperatures = stack(tuple(temperature for _, temperature, _ in states))   # [frames] Scalar+[vertices]
dissipated, conducted, produced = map(stack, zip(*(totals for _, _, totals in states)))   # [frames] Scalar+ each
# The flow's kinetic energy, and the layer's potential energy, minus gravity's drive read on each vertex's place joined
# with its parcel per volume.
kinetic_energies = (drawn_streams * (kinetic * drawn_streams)).sites.sum() / 2   # [frames] Scalar+
vertex_parcels = (volume + (density - expansion * drawn_temperatures) * mass
                  + drawn_temperatures * entropy)                             # [frames] Point-[vertices]
potential_energies = -(cells * (pull & (places & vertex_parcels))).sites.sum()   # [frames] Scalar+
# The energy the budget accounts for: the kinetic and potential energy, with the energy dissipated and less the
# potential energy the heat moved, which stays what it was at the first state drawn; and what it leaves unaccounted, as
# a share of the energy dissipated over the states drawn.
accounted = kinetic_energies + potential_energies + dissipated - conducted   # [frames] Scalar+
unaccounted = (accounted - accounted[0]) / (dissipated[-1] - dissipated[0])  # [frames] Scalar+
render.draw_budget(times, kinetic_energies - kinetic_energies[0], potential_energies - potential_energies[0],
                   dissipated - dissipated[0], conducted - conducted[0], unaccounted, produced - produced[0]);

The top chart shows, against time from the first state drawn, the change of the flow's kinetic energy, in blue, and of the layer's potential energy, in red, the energy viscosity has dissipated, in grey, and minus the potential energy the heat has moved, in orange; the four add up to nothing at every state. While the rolls overturn the layer, its potential energy drops and the dissipated energy climbs; once they settle, the walls keep the layer top-heavy, warming the floor and cooling the ceiling, and the potential energy the heat moves in is what viscosity dissipates. The middle chart shows what the four leave unaccounted, as a share of the energy viscosity dissipates over the states drawn: below a part in a trillion, the round-off of the floating-point numbers. The bottom chart shows the entropy produced, almost all of it by the heat conducted across the layer: it climbs steadily while the layer conducts, steeper as the rolls thin the layers at the floor and the ceiling where conduction hands the heat on, and only ever rises.

In [ ]:
# checks
# Gravity pulls a parcel's mass down and passes over its volume and its entropy.
pull_on_parcel = (mv.y & (-mv.ym).commutator(mass + volume + entropy)).to_array()
np.testing.assert_allclose(pull_on_parcel, -1.0, rtol=1e-12)
# Turned over, the layer's middle half is all but even in temperature on average, the drop in the thin layers at the
# floor and the ceiling; conducting, it drops half a degree.
layer_heights = (mv.y & vertices).to_array()
temperatures = (opening * states[-1][1]).to_array()
middle_drop = temperatures[np.isclose(layer_heights, 0.25)].mean() - temperatures[np.isclose(layer_heights, 0.75)].mean()
assert abs(middle_drop) < 0.05
# The energy budget closes to round-off, as a share of the energy dissipated, and the entropy produced never falls.
assert np.abs(unaccounted.to_array()).max() < 1e-10
assert ((produced[1:] - produced[:-1]) >= 0).all()